# Strategy Construction & OOS Evaluation

End-to-end documentation of our event-driven, vol-targeted strategy.  
OOS period: **2021-10-21 → 2022-06-29** (180 trading days, 11 instruments).

Portfolio construction follows **StrategyWeights lecture, slides 38–43**:

| Convention | Specification |
|---|---|
| Returns | Simple: $r_{t,k} = (P_{t,k} - P_{t-1,k}) / P_{t-1,k}$ |
| EWMA vol | $\lambda=2/(\text{span}+1)$; $\mu_t=\lambda r_t+(1-\lambda)\mu_{t-1}$; $\sigma^2_t=\lambda(r_t-\mu_t)^2+(1-\lambda)\sigma^2_{t-1}$; $\hat{\sigma}=\sigma_{\text{daily}}\times\sqrt{252}$; floor 2%; span=60 |
| Weight | $w_{t,k}=\hat{y}_{t,k}\cdot\sigma_{\text{tgt}}/\hat{\sigma}_{t,k}$, clipped $\pm10$; $\sigma_{\text{tgt}}=10\%$ |
| Lag | $w_t$ uses data $\leq t$, earns $r_{t+1}$ |
| Aggregate | $R^{\text{port}}_{t+1}=\frac{1}{K}\sum_{k=1}^{K}w_{t,k}\,r_{t+1,k}$, $K=11$ (flat=cash) |
| Costs | 2 bps half-spread $+$ 10 bps $\times|\Delta w|$ (Grinold-Kahn) |

In [ ]:
%matplotlib inline
import io, sys, warnings, contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

# ── Robust repo-root detection (works from any CWD, including VS Code) ──────
def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').is_dir() and (p / 'pyproject.toml').is_file():
            return p
    raise FileNotFoundError(f'Cannot find repo root from {start}')

try:
    # VS Code sets __vsc_ipynb_file__ to the notebook's absolute path
    _start = Path(__vsc_ipynb_file__).parent
except NameError:
    _start = Path.cwd()

REPO = _find_repo_root(_start)
sys.path.insert(0, str(REPO / 'src'))

from stml.io import load_data, load_returns_panel
from stml.new_work import config
from stml.new_work.weights import load_oos_events, make_weights
from stml.new_work.targeting import build_vol_panel
from stml.new_work.vol import ewma_close
from stml.new_work.data import load_oof_probabilities
from stml.experimental.backtest import barrier_backtest, performance_metrics
from stml.experimental.significance import significance_report

EVAL_DIR = REPO / 'results' / 'strategy_eval'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repo         : {REPO}')
print(f'BOUNDARY     : {config.BOUNDARY.date()}  (GLOBAL_CUT)')
print(f'VOL_METHOD   : {config.VOL_METHOD}(span={config.EWMA_SPAN})  floor={config.VOL_FLOOR:.0%}')
print(f'σ_tgt        : {config.SIGMA_TGT:.0%}   max_leverage={config.MAX_LEVERAGE}')

## 1. Universe & Data

11 front-month futures across three asset classes.  
Meta-model trained on OOF data from **2020-01-30 → GLOBAL_CUT (2021-10-06)**.  
14-calendar-day embargo. OOS starts **2021-10-21**.

In [ ]:
ohlcv, signals = load_data()
returns_panel  = load_returns_panel(kind='simple')
oos_events     = load_oos_events()
oof_df         = load_oof_probabilities()

INSTRUMENTS = sorted(oos_events['instrument'].unique())
ASSET_CLASS = {
    'es1s': 'Equity', 'nq1s': 'Equity', 'fesx1s': 'Equity',
    'cl1s': 'Energy', 'ho1s': 'Energy', 'rb1s': 'Energy', 'ng1s': 'Energy',
    'gc1s': 'Metals', 'si1s': 'Metals', 'hg1s': 'Metals', 'pl1s': 'Metals',
}
INST_NAME = {
    'es1s': 'ES (S&P 500)', 'nq1s': 'NQ (Nasdaq)', 'fesx1s': 'FESX (Euro Stoxx)',
    'cl1s': 'CL (Crude Oil)', 'ho1s': 'HO (Heating Oil)',
    'rb1s': 'RB (RBOB Gas)', 'ng1s': 'NG (Nat Gas)',
    'gc1s': 'GC (Gold)', 'si1s': 'SI (Silver)',
    'hg1s': 'HG (Copper)', 'pl1s': 'PL (Platinum)',
}

print(f'OOS period : {oos_events["t_start"].min().date()} → {oos_events["t_start"].max().date()}')
print(f'OOS events : {len(oos_events)}  |  Instruments: {len(INSTRUMENTS)}')
print()

summary = (
    oos_events.groupby('instrument')
    .agg(
        n_events  = ('ret', 'count'),
        mean_ret  = ('ret', 'mean'),
        hit_rate  = ('ret', lambda s: (s > 0).mean()),
        frac_long = ('side', lambda s: (s > 0).mean()),
    )
    .assign(
        Class     = lambda df: df.index.map(ASSET_CLASS),
        Instrument= lambda df: df.index.map(INST_NAME),
    )
    [['Class', 'Instrument', 'n_events', 'mean_ret', 'hit_rate', 'frac_long']]
    .rename(columns={
        'n_events': 'Events', 'mean_ret': 'Mean ret',
        'hit_rate': 'Hit rate', 'frac_long': 'Frac long',
    })
    .sort_values(['Class', 'Instrument'])
)
summary['Mean ret']  = summary['Mean ret'].map('{:+.3%}'.format)
summary['Hit rate']  = summary['Hit rate'].map('{:.0%}'.format)
summary['Frac long'] = summary['Frac long'].map('{:.0%}'.format)
print(summary.to_string(index=False))

## 2. Strategy Construction

### Step 1 — EWMA Ex-Ante Volatility

At each bar $t$, we compute a **causal** exponentially weighted variance using simple returns $r_t = P_t/P_{t-1}-1$:

$$\lambda = \frac{2}{\text{span}+1}, \qquad
\mu_t = \lambda r_t + (1-\lambda)\mu_{t-1}, \qquad
\sigma^2_t = \lambda(r_t-\mu_t)^2 + (1-\lambda)\sigma^2_{t-1}$$

Annualised: $\hat{\sigma}_{t,k} = \sigma_t\times\sqrt{252}$, floored at 2%, span = 60.

### Step 2 — Vol-Targeted Weight

Conviction $\hat{y}_{t,k}\in[-1,+1]$ comes from:
- **Method A (Benchmark):** $\hat{y} = \text{side}$ — always follow the primary signal
- **Method B (Meta-filtered):** $\hat{y} = \text{side}\cdot\mathbf{1}[\hat{p}>0.5]$ — only trade when the meta-model is confident

$$w_{t,k} = \hat{y}_{t,k} \cdot \frac{\sigma_{\text{tgt}}}{\hat{\sigma}_{t,k}}, \qquad \sigma_{\text{tgt}}=10\%, \quad w_{t,k}\in[-10,+10]$$

### Step 3 — Portfolio Return

Weight $w_t$ is decided at the close of bar $t$ (signal observed) and earns bar $t+1$'s return. Flat instruments count as cash (K=11 constant in denominator):

$$R^{\text{port}}_{t+1} = \frac{1}{K}\sum_{k=1}^{K}w_{t,k}\,r_{t+1,k} - \frac{1}{K}\sum_k c_k|\Delta w_{t,k}|$$

In [ ]:
DEMO  = {'cl1s': 'CL (Crude Oil)', 'es1s': 'ES (S&P 500)', 'gc1s': 'GC (Gold)'}
COLORS_DEMO = {'cl1s': '#D65F5F', 'es1s': '#4878CF', 'gc1s': '#6ACC65'}

fig, ax = plt.subplots(figsize=(10, 3.5))

for inst, name in DEMO.items():
    sub = ohlcv[ohlcv['instrument'] == inst].set_index('date').sort_index()
    vol = ewma_close(sub['close'], span=config.EWMA_SPAN)
    ax.plot(vol.index, vol * 100, label=name, color=COLORS_DEMO[inst], linewidth=1.4)

ax.axvspan(pd.Timestamp('2021-10-21'), pd.Timestamp('2022-06-29'),
           alpha=0.10, color='steelblue', zorder=0, label='OOS window')
ax.axvline(config.BOUNDARY, color='black', linestyle='--',
           linewidth=0.9, label=f'GLOBAL_CUT ({config.BOUNDARY.date()})')

ax.set_ylabel('Annualised vol (%)')
ax.set_title('EWMA(60) Annualised Volatility — lecture recurrence (simple returns)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(fontsize=8, ncol=2)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## 3. Meta-Model Filter (Method B)

The CPCV-trained meta-model outputs $\hat{p}\in(0,1)$ — the estimated probability that the primary signal is worth following.  
**All-or-nothing filter:** skip the event entirely when $\hat{p}\leq0.5$, reducing 1373 → 745 events (54% taken).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

p_hat = oos_events['calibrated_proba'].dropna()
axes[0].hist(p_hat, bins=30, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].axvline(0.5, color='crimson', linestyle='--', linewidth=1.2, label='Threshold 0.5')
taken = int((p_hat > 0.5).sum())
axes[0].text(0.55, 0.88, f'Taken: {taken}/{len(p_hat)} ({taken/len(p_hat):.0%})',
             transform=axes[0].transAxes, fontsize=9, color='crimson')
axes[0].set_xlabel('Calibrated probability $\\hat{p}$')
axes[0].set_ylabel('Event count')
axes[0].set_title('OOS Meta-Model Probability Distribution')
axes[0].legend(fontsize=8)

evts = oos_events.dropna(subset=['calibrated_proba', 'ret']).copy()
evts['taken'] = evts['calibrated_proba'] > 0.5
for flag, lbl, col in [(True, f'Taken p̂>0.5 (n={evts["taken"].sum()})', 'steelblue'),
                        (False, f'Skipped p̂≤0.5 (n={(~evts["taken"]).sum()})', 'lightcoral')]:
    axes[1].hist(evts.loc[evts['taken'] == flag, 'ret'], bins=30,
                 alpha=0.65, color=col, label=lbl, edgecolor='white', linewidth=0.3)
axes[1].set_xlabel('Side-adjusted event return')
axes[1].set_ylabel('Count')
axes[1].set_title('Event Returns by Meta-Model Decision')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Mean side-adjusted return by decision:')
for flag, lbl in [(True, 'Taken  (p̂>0.5)'), (False, 'Skipped (p̂≤0.5)')]:
    m = evts.loc[evts['taken'] == flag, 'ret'].mean()
    print(f'  {lbl}: {m:+.4%}')

## 4. OOS Evaluation

Both methods share the same vol panel, backtest engine, and cost model.

In [ ]:
# Build shared vol panel
vol_panel = build_vol_panel(
    ohlcv, method=config.VOL_METHOD,
    window=config.LOOKBACK_L, span=config.EWMA_SPAN,
)

METHOD_LABEL = {'A': 'A — Benchmark', 'B-aon': 'B — Meta-filtered'}
METHOD_COLOR = {'A': '#4878CF', 'B-aon': '#D65F5F'}

reports = {}
for method in ['A', 'B-aon']:
    # Suppress make_weights progress prints
    with contextlib.redirect_stdout(io.StringIO()):
        events_m, weights_m = make_weights(
            method, oos_events=oos_events, vol_panel=vol_panel,
        )
    reports[method] = barrier_backtest(events_m, weights_m, returns_panel)
    m = reports[method].metrics
    n_active = int((weights_m.abs() > 0).sum())
    print(f'{METHOD_LABEL[method]:<25}  '
          f'Sharpe={m["sharpe"]:.3f}  '
          f'ann_ret={m["ann_return"]:+.1%}  '
          f'ann_vol={m["ann_vol"]:.1%}  '
          f'max_DD={m["max_dd"]:.1%}  '
          f'events={n_active}/1373')

In [ ]:
fig, (ax_cum, ax_dd) = plt.subplots(
    2, 1, figsize=(11, 6),
    gridspec_kw={'height_ratios': [3, 1]},
    sharex=True,
)

for method in ['A', 'B-aon']:
    r   = reports[method].net_returns
    cum = (1 + r).cumprod() - 1
    eq  = (1 + r).cumprod()
    dd  = (eq / eq.cummax() - 1) * 100

    ax_cum.plot(cum.index, cum * 100,
                label=f'{METHOD_LABEL[method]}  ({cum.iloc[-1]*100:+.1f}%)',
                color=METHOD_COLOR[method], linewidth=1.8)
    ax_dd.fill_between(dd.index, dd, 0, alpha=0.30, color=METHOD_COLOR[method])
    ax_dd.plot(dd.index, dd, color=METHOD_COLOR[method], linewidth=0.9)

ax_cum.axhline(0, color='black', linewidth=0.6, linestyle='--')
ax_cum.set_ylabel('Cumulative net return (%)')
ax_cum.set_title(
    'OOS Cumulative Net Returns  (2021-10-21 → 2022-06-29)',
    fontsize=11, fontweight='bold',
)
ax_cum.legend(fontsize=9, loc='upper left')
ax_cum.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

ax_dd.set_ylabel('Drawdown (%)')
ax_dd.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax_dd.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax_dd.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.setp(ax_dd.get_xticklabels(), rotation=30, ha='right')

fig.align_ylabels()
plt.tight_layout()

out_path = EVAL_DIR / 'cumulative_returns.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path.relative_to(REPO)}')

In [ ]:
# ── Portfolio-level performance table ────────────────────────────────────────
rows = []
for method in ['A', 'B-aon']:
    r  = reports[method].net_returns
    m  = reports[method].metrics
    sr = significance_report(r.values, n_boot=2000, seed=42)
    n_active = int((reports[method].weights.fillna(0).abs().sum(axis=1) > 0).sum())
    rows.append({
        'Method':             METHOD_LABEL[method],
        'Ann return':         f"{m['ann_return']:+.2%}",
        'Ann vol':            f"{m['ann_vol']:.2%}",
        'Sharpe':             f"{m['sharpe']:.3f}",
        't-stat':             f"{sr.t_stat:.2f}",
        'SR 95% CI':          f"[{sr.bootstrap_ci_low*252**0.5:.2f}, {sr.bootstrap_ci_high*252**0.5:.2f}]",
        'PSR(SR*=0)':         f"{sr.psr_zero:.3f}",
        'Max DD':             f"{m['max_dd']:.2%}",
        'Total cost (bps)':   f"{m['total_cost_bps']:.0f}",
        'Ann turnover':       f"{m['turnover_per_year']:.2f}",
        'Breakeven ½-spread': f"{m['total_cost_bps']/m['turnover_per_year']*m['ann_vol']/m['ann_vol']:.1f} bps"
            if m['turnover_per_year'] > 0 else '—',
        'Active days':        f"{n_active}/180",
    })

perf = pd.DataFrame(rows).set_index('Method').T
print(perf.to_string())

## 5. Performance by Asset Class & Instrument

We decompose portfolio returns into per-instrument contributions:
$\text{contrib}_{t,k} = w_{t,k} \cdot r_{t+1,k}$ (before the $1/K$ averaging).

Positive annualised contribution means the instrument helped; Sharpe on contributions indicates consistency.

In [ ]:
# Per-instrument contribution stats
rows_inst = []
for method in ['A', 'B-aon']:
    w_panel = reports[method].weights              # date × instrument
    r_panel = returns_panel.reindex(w_panel.index).reindex(columns=w_panel.columns)
    fwd_r   = r_panel.shift(-1).fillna(0.0)        # forward return (lecture lag)

    for inst in w_panel.columns:
        contrib = (w_panel[inst] * fwd_r[inst]).iloc[:-1]   # drop last (no fwd)

        # Event-level stats
        inst_evts = oos_events[oos_events['instrument'] == inst].copy()
        if method == 'B-aon':
            inst_evts = inst_evts[inst_evts['calibrated_proba'] > 0.5]
        n_evts   = len(inst_evts)
        hit_rate = float((inst_evts['ret'] > 0).mean()) if n_evts > 0 else float('nan')

        # Portfolio contribution metrics
        m_c = performance_metrics(contrib)
        rows_inst.append({
            'Method':          METHOD_LABEL[method],
            'Class':           ASSET_CLASS.get(inst, '?'),
            'Instrument':      INST_NAME.get(inst, inst),
            'Events taken':    n_evts,
            'Hit rate':        f'{hit_rate:.0%}' if not np.isnan(hit_rate) else '—',
            'Ann contrib (%)': round(m_c['ann_return'] * 100, 2),
            'Sharpe (contrib)': round(m_c['sharpe'], 2) if np.isfinite(m_c['sharpe']) else float('nan'),
            'Max DD':          f"{m_c['max_dd']:.1%}",
        })

inst_df = pd.DataFrame(rows_inst)

# ── Table: per instrument, side by side for A and B ─────────────────────────
for method_label in [METHOD_LABEL['A'], METHOD_LABEL['B-aon']]:
    sub = inst_df[inst_df['Method'] == method_label].drop(columns='Method')
    sub = sub.sort_values(['Class', 'Instrument']).reset_index(drop=True)
    print(f'\n{'─'*70}')
    print(f'  {method_label}')
    print(f'{'─'*70}')
    print(sub.to_string(index=False))

In [ ]:
# ── Bar chart: ann contribution per instrument, A vs B ───────────────────────
CLASS_COLOR = {'Equity': '#4878CF', 'Energy': '#D65F5F', 'Metals': '#6ACC65'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, method in zip(axes, ['A', 'B-aon']):
    sub = inst_df[inst_df['Method'] == METHOD_LABEL[method]].copy()
    sub = sub.sort_values('Ann contrib (%)')
    colors = [CLASS_COLOR.get(c, 'grey') for c in sub['Class']]
    bars = ax.barh(sub['Instrument'], sub['Ann contrib (%)'],
                   color=colors, edgecolor='white', linewidth=0.4)
    ax.axvline(0, color='black', linewidth=0.7)
    ax.set_xlabel('Annualised contribution (%)')
    ax.set_title(METHOD_LABEL[method], fontsize=9, fontweight='bold')

    # Label bars
    for bar, val in zip(bars, sub['Ann contrib (%)']):
        xpos = bar.get_width() + (0.05 if val >= 0 else -0.05)
        ha   = 'left' if val >= 0 else 'right'
        ax.text(xpos, bar.get_y() + bar.get_height()/2,
                f'{val:+.1f}%', va='center', ha=ha, fontsize=7.5)

legend_handles = [Patch(color=c, label=l) for l, c in CLASS_COLOR.items()]
axes[1].legend(handles=legend_handles, fontsize=8, loc='lower right')

fig.suptitle('Annualised Return Contribution by Instrument (OOS, before 1/K)',
             fontsize=10, fontweight='bold')
plt.tight_layout()

out_path2 = EVAL_DIR / 'per_instrument_contribution.png'
fig.savefig(out_path2, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path2.relative_to(REPO)}')

In [ ]:
# ── Per-asset-class aggregated table ─────────────────────────────────────────
print('Per-asset-class summary (sum of instrument contributions)\n')
for method_label in [METHOD_LABEL['A'], METHOD_LABEL['B-aon']]:
    sub = inst_df[inst_df['Method'] == method_label].copy()
    agg = (
        sub.groupby('Class')
        .agg(
            Instruments    = ('Instrument', 'count'),
            Events_taken   = ('Events taken', 'sum'),
            Ann_contrib_pct= ('Ann contrib (%)', 'sum'),
        )
        .rename(columns={
            'Instruments': 'N insts',
            'Events_taken': 'Events',
            'Ann_contrib_pct': 'Ann contrib (%)',
        })
    )
    agg['Ann contrib (%)'] = agg['Ann contrib (%)'].map('{:+.2f}%'.format)
    print(f'  {method_label}')
    print(agg.to_string())
    print()

## 6. Summary

| | Benchmark (A) | Meta-filtered (B) | Δ |
|---|---|---|---|
| Events traded | 1373 / 1373 | 745 / 1373 | −46% |
| OOS Sharpe | 1.41 | **1.71** | +0.30 |
| Ann return (net) | +8.6% | +4.9% | −3.7 pp |
| Ann vol | 6.1% | 2.9% | −3.2 pp |
| Max drawdown | −3.4% | −1.4% | +2.0 pp |
| Total costs | 416 bps | 188 bps | −55% |

The meta-model filter improves risk-adjusted performance by selectively sitting out the lower-confidence half of events. The absolute return drops because uninvested days earn 0 (cash); the drawdown and cost both halve. Across asset classes, Energy is the dominant contributor in both methods; Equity is the largest drag.

**Statistical caveat:** t-stats of 1.2–1.4 over 180 trading days are not conventionally significant at 5%, but PSR(SR*=0) ≈ 0.86–0.91 indicates meaningful evidence of positive Sharpe. The hidden test set (Jul–Dec 2022) has not been accessed.